# ICR Probe

Measure how closely each layer's hidden-state update follows attention, then fit a small classifier. The experiment adapts the method to token-level code defect detection. [Paper](https://arxiv.org/abs/2507.16488)

Train an independent probe on each benchmark. Line and sample scores use the maximum annotated token score. F1 thresholds come from validation.

In [1]:
from pathlib import Path
import sys

if Path("traces.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd
import torch
from IPython.display import display

from experiments.data import DATASETS, PILOT_LIMITS
from experiments.evaluation import TraceStore, evaluate
from experiments.methods import ICRProbe

pilot = False
limits = PILOT_LIMITS if pilot else None
torch.set_num_threads(min(4, torch.get_num_threads()))
store = TraceStore(quantize=True)

In [3]:
results = []
for benchmark in DATASETS:
    splits = store.load_splits(benchmark, limits)
    results.extend(
        {"benchmark": benchmark, **row}
        for row in evaluate(ICRProbe(), splits)
    )

metrics = pd.DataFrame(results).set_index(["benchmark", "level"])
display(metrics.round(4))

n  positives  positive_rate  pr_auc  \
benchmark      level                                            
mbpp           token   3373        660         0.1957  0.1742   
               line     509         54         0.1061  0.1429   
               sample    74         31         0.4189  0.4265   
humaneval      token   3714        665         0.1791  0.3936   
               line     484         55         0.1136  0.3562   
               sample    46         19         0.4130  0.3586   
humaneval_java token   1157        205         0.1772  0.1715   
               line     155         15         0.0968  0.1435   
               sample    36         11         0.3056  0.2529   
defects4j      token   3901       2220         0.5691  0.7797   
               line     448        148         0.3304  0.5807   
               sample    86         35         0.4070  0.5425   

                       average_precision  roc_auc  precision  recall      f1  \
benchmark      level                                                           
mbpp           token              0.1751   0.4392     0.1994  0.9879  0.3318   
               line               0.1520   0.5404     0.1106  0.9630  0.1985   
               sample             0.4453   0.5135     0.4203  0.9355  0.5800   
humaneval      token              0.3945   0.7079     0.3772  0.5729  0.4549   
               line               0.3613   0.6693     0.3860  0.4000  0.3929   
               sample             0.3739   0.3275     0.3043  0.3684  0.3333   
humaneval_java token              0.1732   0.5110     0.1741  0.3415  0.2306   
               line               0.1569   0.6831     0.0000  0.0000  0.0000   
               sample             0.2859   0.4273     0.3030  0.9091  0.4545   
defects4j      token              0.7800   0.7661     0.6841  0.8788  0.7693   
               line               0.5841   0.7909     0.5829  0.7365  0.6507   
               sample             0.5519   0.6056     0.4211  0.9143  0.5766   

                       threshold  
benchmark      level              
mbpp           token      0.1917  
               line       0.1971  
               sample     0.2016  
humaneval      token      0.1552  
               line       0.1584  
               sample     0.1565  
humaneval_java token      0.1731  
               line       0.1769  
               sample     0.1745  
defects4j      token      0.4703  
               line       0.4722  
               sample     0.4721